Analise exploratório CLA- OMNIBOX

**Objetivo:** Padronização dos dados para o formato '.parquet', exploração e limpeza dos dados. 

**Motor de Processamento:** DuckDB (Processamento Analítico em Memória)

**Visualização:** Pandas e Seaborn (Apenas para agregações e amostras)

### Etapa 1 - Setup e Configurações Iniciais
Importação de bilbiotecas, setup e conexão com o DuckDB e configuração de pastas de trabalho.

In [1]:
# Criar o kernel no Poetry:  
# Garante que o modulo externo seja sempre recarregado 
%load_ext autoreload
%autoreload 2

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path
import os 
import shutil
import cla_omnibox_tratamento_funcoes as omni_val
import json
import ipywidgets as widgets
from tqdm import tqdm

import sys 

print(10)
# Configurações visuais padronizadas para os gráficos e tabelas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
sns.set_theme(style="whitegrid")

# 1. Inicializando a conexão com o DuckDB (em memória). 
con = duckdb.connect()

# 2. Otimizações de Hardware 
# Limitação do uso de memória RAM para evitar problemas. 
con.execute("PRAGMA memory_limit='6GB'") 

#  Limitação do numero de threads
con.execute("PRAGMA threads=8") 

print("Bibliotecas carregadas e motor DuckDB configurado com sucesso!")

# Configuração das pastas de trabalho (se não existirem serão criadas)

DIR_BASE_APP = Path.cwd() 
DIR_BASE = DIR_BASE_APP.parents[0]
PATH_CONFIG = DIR_BASE / "config"
FILE_JSON_CONFIG = "cla_omnibox_schema_dados.json"
caminho_completo_json = PATH_CONFIG / FILE_JSON_CONFIG
DIR_RAW = DIR_BASE / "dados" / "raw"
DIR_BRONZE = DIR_BASE / "dados" / "bronze"
DIR_PRATA = DIR_BASE / "dados" /  "prata"
DIR_GA = f"{DIR_BRONZE}/google_analytics/*.parquet" 
print(f"Pastas e caminhos de configuração definidos com sucesso!") 

10
Bibliotecas carregadas e motor DuckDB configurado com sucesso!
Pastas e caminhos de configuração definidos com sucesso!


### Etapa 2 - Carga da estrutura de dados 
Carregar json com o esquema de dados a ser validade e alguns parametros de validação. 

In [2]:

print(f"Carregando a estrutura das tabelas (schema)")
try: 

    caminho_completo_json = PATH_CONFIG / FILE_JSON_CONFIG
    with open(caminho_completo_json, 'r', encoding='utf-8') as f:
            config_mestre = json.load(f)

    tabelas_mapeadas = [t['tabela'] for t in config_mestre['esquema']]
    print(f"Tabelas configuradas no esquema: {tabelas_mapeadas}")

    print(f"Sucesso: JSON mestre carregado a partir de {caminho_completo_json}")

except FileNotFoundError:
    print(f"Erro: O arquivo {FILE_JSON_CONFIG} não foi encontrado no caminho {PATH_CONFIG}")
except json.JSONDecodeError:
    print(f"Erro: O arquivo {FILE_JSON_CONFIG} não está em um formato JSON válido.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Carregando a estrutura das tabelas (schema)
Tabelas configuradas no esquema: ['category', 'sub_category', 'products', 'customers', 'orders', 'order_items', 'mkt_campaigns', 'mkt_actions', 'mkt_spend', 'google_analytics']
Sucesso: JSON mestre carregado a partir de c:\gitHub\cla-omnibox\config\cla_omnibox_schema_dados.json


### Etapa 3  - Mapeamento dos arquivos fisicos (parquet) no DuckDB
Criação de **VIEWS** no DuckDB para mapear os arquivos parquet da camada Bronze. 

In [3]:
# --- Etapa 4: Mapeamento de Metadados e Criacao de Views (Camada Bronze) ---

print("Iniciando mapeamento de metadados para criacao de views...\n")
views_criadas = []
print("inicio") 
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    view_name = tabela_nome 
    caminho_str = None
    origem_log = ""
    
    busca_arquivo = list(DIR_BRONZE.rglob(f"{tabela_nome}.parquet"))
    busca_pasta = [p for p in DIR_BRONZE.rglob(tabela_nome) if p.is_dir()]
    
    if busca_arquivo:
        caminho_str = busca_arquivo[0].as_posix()
        origem_log = busca_arquivo[0].name
    elif busca_pasta:
        pasta_encontrada = busca_pasta[0]
        if list(pasta_encontrada.glob("*.parquet")):
            caminho_str = f"{pasta_encontrada.as_posix()}/*.parquet"
            origem_log = f"Pasta particionada: {pasta_encontrada.name}/*.parquet"
        
    if caminho_str:
        # Tratamento dinamico e automatizado para o Google Analytics
        # if tabela_nome == 'google_analytics':
        #     campos_json = [c['nome'] for c in entrada['campos']]
        #     colunas_raiz_ga = ['event_date', 'event_timestamp', 'event_name', 'user_id', 'user_pseudo_id']
        #     linhas_select = []
            
        #     for campo in campos_json:
        #         if campo in colunas_raiz_ga:
        #             # Campo padrao da raiz do arquivo
        #             linhas_select.append(campo)
        #         elif campo.startswith('geo_'):
        #             # Converte "geo_country" para o formato do struct "geo.country"
        #             campo_limpo = campo.replace('geo_', '')
        #             linhas_select.append(f"geo.{campo_limpo} AS {campo}")
        #         else:
        #             # Se nao for raiz nem geo, o script assume dinamicamente que esta no event_params.
        #             # O COALESCE tenta pegar o valor da chave caso seja string_value ou int_value.
        #             extrator = f"COALESCE((list_filter(event_params, x -> x.key == '{campo}')[1]).value.string_value, (list_filter(event_params, x -> x.key == '{campo}')[1]).value.int_value::VARCHAR) AS {campo}"
        #             linhas_select.append(extrator)
            
        #     # Junta todas as linhas geradas em uma unica string separada por virgulas
        #     colunas_dinamicas_sql = ",\n                ".join(linhas_select)
            
        #     query_view = f"""
        #     CREATE OR REPLACE VIEW {view_name} AS 
        #     SELECT 
        #         {colunas_dinamicas_sql}
        #     FROM read_parquet('{caminho_str}', union_by_name=true);
        #     """
        # else:
            # Comportamento padrao para as demais tabelas
        query_view = f"CREATE OR REPLACE VIEW {view_name} AS SELECT * FROM read_parquet('{caminho_str}', union_by_name=true);"
        
        con.execute(query_view)
        views_criadas.append(view_name)
        print(f"View '{view_name}' mapeada com sucesso -> {origem_log}")
    else:
        print(f"Alerta: Arquivo unico ou pasta particionada para '{tabela_nome}' nao encontrados na Bronze.")

if views_criadas:
    print(f"\nSucesso! {len(views_criadas)} views foram disponibilizadas no DuckDB.")

Iniciando mapeamento de metadados para criacao de views...

inicio
View 'category' mapeada com sucesso -> category.parquet
View 'sub_category' mapeada com sucesso -> sub_category.parquet
View 'products' mapeada com sucesso -> products.parquet
View 'customers' mapeada com sucesso -> customers.parquet
View 'orders' mapeada com sucesso -> orders.parquet
View 'order_items' mapeada com sucesso -> order_items.parquet
View 'mkt_campaigns' mapeada com sucesso -> mkt_campaigns.parquet
View 'mkt_actions' mapeada com sucesso -> mkt_actions.parquet
View 'mkt_spend' mapeada com sucesso -> mkt_spend.parquet
View 'google_analytics' mapeada com sucesso -> Pasta particionada: google_analytics/*.parquet

Sucesso! 10 views foram disponibilizadas no DuckDB.


### Etapa 4 - Inspeção de Schema e Volumetria
Verificacao de volumetria e estrutura das tabelas 

In [4]:
# --- Etapa 5: Análise de Volumetria e Validação de Schema (Camada Bronze) ---

print("Calculando volumetria e validando o schema das tabelas...")

dados_volumetria = []

for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    
    # Extrai a lista de nomes das colunas esperadas a partir do JSON Mestre
    campos_esperados = [campo['nome'] for campo in entrada['campos']]
    qtd_colunas_esperadas = len(campos_esperados)
    
    try:
        # 1. Consulta o número total de linhas
        query_linhas = f"SELECT COUNT(*) FROM {tabela_nome}"
        qtd_linhas = con.execute(query_linhas).fetchone()[0]
        
        # 2. Consulta os nomes das colunas físicas mapeadas pelo DuckDB
        query_colunas = f"SELECT column_name FROM information_schema.columns WHERE table_name = '{tabela_nome}'"
        campos_reais = [linha[0] for linha in con.execute(query_colunas).fetchall()]
        qtd_colunas_reais = len(campos_reais)
        
        # 3. Validação de Schema usando Sets (Conjuntos)
        set_esperados = set(campos_esperados)
        set_reais = set(campos_reais)
        
        colunas_faltantes = set_esperados - set_reais # Estão no JSON, mas não no arquivo
        colunas_sobrando = set_reais - set_esperados  # Estão no arquivo, mas não no JSON
        
        # 4. Avalia o status e monta os detalhes
        if not colunas_faltantes and not colunas_sobrando:
            status_schema = "OK"
            detalhes = "-"
        else:
            status_schema = "Divergente"
            lista_detalhes = []
            if colunas_faltantes:
                lista_detalhes.append(f"Falta no arquivo: {', '.join(colunas_faltantes)}")
            if colunas_sobrando:
                lista_detalhes.append(f"Não mapeado no JSON: {', '.join(colunas_sobrando)}")
            detalhes = " | ".join(lista_detalhes)
            
        dados_volumetria.append({
            "Tabela (View)": tabela_nome,
            "Total de Linhas": qtd_linhas,
            "Colunas no Arquivo": qtd_colunas_reais,
            "Colunas no JSON": qtd_colunas_esperadas,
            "Status do Schema": status_schema,
            "Detalhes da Divergência": detalhes
        })
        
    except Exception:
        # Captura o erro caso a View não tenha sido criada na Etapa 4
        dados_volumetria.append({
            "Tabela (View)": tabela_nome,
            "Total de Linhas": "Não mapeada",
            "Colunas no Arquivo": "-",
            "Colunas no JSON": qtd_colunas_esperadas,
            "Status do Schema": "Pendente",
            "Detalhes da Divergência": "Arquivo não encontrado na Bronze"
        })

# Converte para DataFrame do Pandas para uma visualização elegante no Notebook
df_volumetria = pd.DataFrame(dados_volumetria)

# Exibe a tabela formatada
display(df_volumetria)

Calculando volumetria e validando o schema das tabelas...


,Tabela (View),Total de Linhas,Colunas no Arquivo,Colunas no JSON,Status do Schema,Detalhes da Divergência
0,category,5,3,3,OK,-
1,sub_category,20,3,3,OK,-
2,products,91,7,7,OK,-
3,customers,51241,9,9,OK,-
4,orders,48193,14,14,OK,-
5,order_items,83960,7,7,OK,-
6,mkt_campaigns,14,7,7,OK,-
7,mkt_actions,37,9,9,OK,-
8,mkt_spend,4953,5,5,OK,-
9,google_analytics,2835930,14,14,OK,-


### Etapa 5 - Validação de Nulos e Brancos
Verifica integridade de campos que não aceitam valores brancos e nulos. 

In [5]:
# --- Etapa 6: Execução da Validação de Nulos e Brancos ---

print("Iniciando auditoria de nulos e brancos...\n")

lista_resultados_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função do nosso arquivo externo
        df_resultado_tabela = omni_val.validar_nulos_e_brancos(con, tabela_nome, campos)
   
        lista_resultados_df.append(df_resultado_tabela)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a tabela '{tabela_nome}': {e}")

# Concatena os resultados de todas as tabelas em um único DataFrame
if lista_resultados_df:
    df_auditoria_final = pd.concat(lista_resultados_df, ignore_index=True)
    
    # Exibe o relatório final
    display(df_auditoria_final)
else:
    print("Nenhuma validação pôde ser executada.")

Iniciando auditoria de nulos e brancos...



,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,Todas,-,0,Aprovado (Nulos/Brancos)
1,sub_category,Todas,-,0,Aprovado (Nulos/Brancos)
2,products,Todas,-,0,Aprovado (Nulos/Brancos)
3,customers,Todas,-,0,Aprovado (Nulos/Brancos)
4,orders,Todas,-,0,Aprovado (Nulos/Brancos)
5,order_items,Todas,-,0,Aprovado (Nulos/Brancos)
6,mkt_campaigns,Todas,-,0,Aprovado (Nulos/Brancos)
7,mkt_actions,Todas,-,0,Aprovado (Nulos/Brancos)
8,mkt_spend,Todas,-,0,Aprovado (Nulos/Brancos)
9,google_analytics,Todas,-,0,Aprovado (Nulos/Brancos)


### Etapa 6 - Validação de Tipos de dados

In [6]:
print("Iniciando auditoria de tipagem de dados...\n")

lista_resultados_tipos_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a NOVA função do nosso arquivo externo
        df_resultado_tipo = omni_val.validar_tipagem_dados(con, tabela_nome, campos)
        lista_resultados_tipos_df.append(df_resultado_tipo)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar os tipos da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_tipos_df:
    df_auditoria_tipos = pd.concat(lista_resultados_tipos_df, ignore_index=True)
    display(df_auditoria_tipos)
else:
    print("Nenhuma validação de tipos pôde ser executada.")

Iniciando auditoria de tipagem de dados...



,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,Todas,-,0,✅ Aprovado (Tipagem)
1,sub_category,Todas,-,0,✅ Aprovado (Tipagem)
2,products,Todas,-,0,✅ Aprovado (Tipagem)
3,customers,Todas,-,0,✅ Aprovado (Tipagem)
4,orders,Todas,-,0,✅ Aprovado (Tipagem)
5,order_items,Todas,-,0,✅ Aprovado (Tipagem)
6,mkt_campaigns,Todas,-,0,✅ Aprovado (Tipagem)
7,mkt_actions,Todas,-,0,✅ Aprovado (Tipagem)
8,mkt_spend,Todas,-,0,✅ Aprovado (Tipagem)
9,google_analytics,Todas,-,0,✅ Aprovado (Tipagem)


### Etapa 7 - Validação de valores unicos

In [7]:
print("Iniciando auditoria de Valores Únicos...\n")

lista_resultados_unicos_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de unicidade do nosso módulo externo
        df_resultado_unico = omni_val.validar_valores_unicos(con, tabela_nome, campos)
        lista_resultados_unicos_df.append(df_resultado_unico)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a unicidade da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_unicos_df:
    df_auditoria_unicos = pd.concat(lista_resultados_unicos_df, ignore_index=True)
    display(df_auditoria_unicos)
else:
    print("Nenhuma validação de valores únicos pôde ser executada.")

Iniciando auditoria de Valores Únicos...



,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,Todas,-,0,✅ Aprovado (Valores Únicos)
1,sub_category,Todas,-,0,✅ Aprovado (Valores Únicos)
2,products,Todas,-,0,✅ Aprovado (Valores Únicos)
3,customers,Todas,-,0,✅ Aprovado (Valores Únicos)
4,orders,Todas,-,0,✅ Aprovado (Valores Únicos)
5,order_items,Todas,-,0,✅ Aprovado (Valores Únicos)
6,mkt_campaigns,Todas,-,0,✅ Aprovado (Valores Únicos)
7,mkt_actions,Todas,-,0,✅ Aprovado (Valores Únicos)
8,mkt_spend,Todas,-,0,✅ Aprovado (Valores Únicos)
9,google_analytics,Todas,-,0,✅ Aprovado (Valores Únicos)


### Etapa 8 - Validação de regras numéricas (zeros e negativos) 

In [8]:

print("Iniciando auditoria de Qualidade de Dados: Zeros e Negativos...\n")

lista_resultados_num_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação numérica
        df_resultado_num = omni_val.validar_regras_numericas(con, tabela_nome, campos)
        lista_resultados_num_df.append(df_resultado_num)
        
    except duckdb.CatalogException:
        print(f"⚠ Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar as regras numéricas da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_num_df:
    df_auditoria_num = pd.concat(lista_resultados_num_df, ignore_index=True)
    display(df_auditoria_num)
else:
    print("Nenhuma validação numérica pôde ser executada.")

    

Iniciando auditoria de Qualidade de Dados: Zeros e Negativos...



,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,Todas,-,0,✅ Aprovado (Regras Numéricas)
1,sub_category,Todas,-,0,✅ Aprovado (Regras Numéricas)
2,products,Todas,-,0,✅ Aprovado (Regras Numéricas)
3,customers,Todas,-,0,✅ Aprovado (Regras Numéricas)
4,orders,Todas,-,0,✅ Aprovado (Regras Numéricas)
5,order_items,Todas,-,0,✅ Aprovado (Regras Numéricas)
6,mkt_campaigns,Todas,-,0,✅ Aprovado (Regras Numéricas)
7,mkt_actions,Todas,-,0,✅ Aprovado (Regras Numéricas)
8,mkt_spend,Todas,-,0,✅ Aprovado (Regras Numéricas)
9,google_analytics,Todas,-,0,✅ Aprovado (Regras Numéricas)


### Etapa 9 - Validações de formato (via REGEX) 

In [9]:
print("Iniciando auditoria de Qualidade formatos (padrões Regex)...\n")

lista_resultados_regex_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de regex
        df_resultado_regex = omni_val.validar_regex(con, tabela_nome, campos)
        lista_resultados_regex_df.append(df_resultado_regex)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar os formatos regex da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_regex_df:
    df_auditoria_regex = pd.concat(lista_resultados_regex_df, ignore_index=True)
    display(df_auditoria_regex)
else:
    print("Nenhuma validação de Regex pôde ser executada.")

Iniciando auditoria de Qualidade formatos (padrões Regex)...



,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,Todas,-,0,Aprovado (Formato/Regex)
1,sub_category,Todas,-,0,Aprovado (Formato/Regex)
2,products,Todas,-,0,Aprovado (Formato/Regex)
3,customers,Todas,-,0,Aprovado (Formato/Regex)
4,orders,Todas,-,0,Aprovado (Formato/Regex)
5,order_items,Todas,-,0,Aprovado (Formato/Regex)
6,mkt_campaigns,Todas,-,0,Aprovado (Formato/Regex)
7,mkt_actions,Todas,-,0,Aprovado (Formato/Regex)
8,mkt_spend,Todas,-,0,Aprovado (Formato/Regex)
9,google_analytics,Todas,-,0,Aprovado (Formato/Regex)


### Etapa 10 - Validações de lista de valores 

In [10]:
print("Iniciando auditoria de Qualidade de Dados: Lista de Valores de Domínio...\n")

lista_resultados_valores_df = []

# Itera sobre todas as tabelas do nosso JSON Mestre
for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de listas
        df_resultado_valores = omni_val.validar_lista_valores(con, tabela_nome, campos)
        lista_resultados_valores_df.append(df_resultado_valores)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a lista de valores da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_valores_df:
    df_auditoria_valores = pd.concat(lista_resultados_valores_df, ignore_index=True)
    display(df_auditoria_valores)
else:
    print("Nenhuma validação de lista de valores pôde ser executada.")

Iniciando auditoria de Qualidade de Dados: Lista de Valores de Domínio...

Erro ao validar a lista de valores da tabela 'customers': 'list' object has no attribute 'split'


,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,Todas,-,0,✅ Aprovado (Lista de Validação)
1,sub_category,Todas,-,0,✅ Aprovado (Lista de Validação)
2,products,Todas,-,0,✅ Aprovado (Lista de Validação)
3,orders,Todas,-,0,✅ Aprovado (Lista de Validação)
4,order_items,Todas,-,0,✅ Aprovado (Lista de Validação)
5,mkt_campaigns,Todas,-,0,✅ Aprovado (Lista de Validação)
6,mkt_actions,Todas,-,0,✅ Aprovado (Lista de Validação)
7,mkt_spend,Todas,-,0,✅ Aprovado (Lista de Validação)
8,google_analytics,Todas,-,0,✅ Aprovado (Lista de Validação)


### Etapa 11 Validações de integridade de Chaves primárias (PK) 

In [11]:
print("Iniciando auditoria de integridade de Chaves Primárias (Simples e Compostas)...\n")

lista_resultados_pk_df = []

for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de PK
        df_resultado_pk = omni_val.validar_chave_primaria(con, tabela_nome, campos)
        lista_resultados_pk_df.append(df_resultado_pk)
        
    except duckdb.CatalogException:
        print(f"Tabela/View '{tabela_nome}' não encontrada no DuckDB. Pulando validação.")
    except Exception as e:
        print(f"Erro ao validar a PK da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_pk_df:
    df_auditoria_pk = pd.concat(lista_resultados_pk_df, ignore_index=True)
    display(df_auditoria_pk)
else:
    print("Nenhuma validação de PK pôde ser executada.")

Iniciando auditoria de integridade de Chaves Primárias (Simples e Compostas)...



,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,category_id,-,0,✅ Aprovado (Chave Primária)
1,sub_category,sub_category_id,-,0,✅ Aprovado (Chave Primária)
2,products,product_id,-,0,✅ Aprovado (Chave Primária)
3,customers,customer_id,-,0,✅ Aprovado (Chave Primária)
4,orders,order_id,-,0,✅ Aprovado (Chave Primária)
5,order_items,order_item_id,-,0,✅ Aprovado (Chave Primária)
6,mkt_campaigns,campaign_id,-,0,✅ Aprovado (Chave Primária)
7,mkt_actions,action_id + campaign_id,-,0,✅ Aprovado (Chave Primária)
8,mkt_spend,date + action_id,-,0,✅ Aprovado (Chave Primária)
9,google_analytics,-,-,0,⚠️ Nenhuma PK configurada


### Etapa 12 - Validações de integridade de Chaves estrangeiras (FK) 

In [12]:

print("Iniciando auditoria de integridade de Chaves Estrangeiras...\n")

lista_resultados_fk_df = []

for entrada in config_mestre['esquema']:
    tabela_nome = entrada['tabela']
    campos = entrada['campos']
    
    try:
        # Chama a função de validação de FK
        df_resultado_fk = omni_val.validar_chaves_estrangeiras(con, tabela_nome, campos)
        lista_resultados_fk_df.append(df_resultado_fk)
        
    except duckdb.CatalogException as e:
        print(f"Erro de Catálogo: Certifique-se de que todas as tabelas relacionadas foram carregadas. Detalhe: {e}")
    except Exception as e:
        print(f"Erro ao validar FK da tabela '{tabela_nome}': {e}")

# Concatena e exibe os resultados
if lista_resultados_fk_df:
    df_auditoria_fk = pd.concat(lista_resultados_fk_df, ignore_index=True)
    display(df_auditoria_fk)
else:
    print("Nenhuma validação de FK pôde ser executada.")

Iniciando auditoria de integridade de Chaves Estrangeiras...



,Tabela,Coluna,Regra Violada,Qtd Erros,Status
0,category,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
1,sub_category,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
2,products,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
3,customers,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
4,orders,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
5,order_items,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
6,mkt_campaigns,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
7,mkt_actions,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
8,mkt_spend,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)
9,google_analytics,Todas,-,0,✅ Aprovado (Chaves Estrangeiras)


### 13 - Validação de regra de negocio: Consistencia do total da venda

In [15]:
def gerar_descricao_estatistica(con: duckdb.DuckDBPyConnection, nome_tabela: str) -> None:
    """
    Gera e exibe o resumo estatístico (numérico e categórico) de uma tabela.
    Ideal para identificar dispersão, outliers e frequência de categorias.
    """
    print(f"\n=== Resumo Estatístico: Tabela {nome_tabela.upper()} ===")
    
    # Carrega os dados da tabela para um DataFrame Pandas
    # Nota: Para tabelas gigantescas, pode-se usar LIMIT ou amostragem
    df = con.execute(f"SELECT * FROM {nome_tabela}").to_df()
    
    # 1. Resumo Numérico (Contagem, Média, Std, Min, Max, Percentis)
    desc_num = df.describe()
    if not desc_num.empty:
        print("\n[Dados Numéricos]")
        display(desc_num)
    
    # 2. Resumo Categórico (Contagem, Únicos, Top, Frequência)
    # include=['O'] foca em objetos/strings
    desc_cat = df.describe(include=['O'])
    if not desc_cat.empty:
        print("\n[Dados Categóricos / Texto]")
        display(desc_cat)
        
    print("-" * 50)
print("Iniciando auditoria de Regras de Negocio Customizadas...\n")

resultados_negocio = []

# 1. Definicao da Regra: Consistencia de Total da Venda
nome_regra_venda = "Consistencia de Total da Venda (Itens vs Cabecalho)"

query_venda_itens = """
    WITH SomaItens AS (
        SELECT 
            order_id, 
            SUM(price * quantity) as valor_bruto_itens
        FROM order_items
        GROUP BY order_id
    ),
    Auditoria AS (
        SELECT 
            v.order_id,
            v.gross_amount as valor_registrado,
            (i.valor_bruto_itens - COALESCE(v.discount, 0)) as valor_calculado
        FROM orders v
        JOIN SomaItens i ON v.order_id = i.order_id
        WHERE v.order_status = 'ÇOMPLETED'
    )
    SELECT COUNT(*) 
    FROM Auditoria 
    WHERE ROUND(valor_registrado, 2) <> ROUND(valor_calculado, 2)
"""

# Chamada da funcao generica
df_regra_venda = omni_val.validar_regra_negocio_sql(con, nome_regra_venda, query_venda_itens)
resultados_negocio.append(df_regra_venda)

# Consolidacao e exibicao
if resultados_negocio:
    df_auditoria_negocio = pd.concat(resultados_negocio, ignore_index=True)
    display(df_auditoria_negocio)
else:
    print("Nenhuma regra de negocio foi validada.")

Iniciando auditoria de Regras de Negocio Customizadas...



,Validacao de Negocio,Status,Qtd Registros com Erro
0,Consistencia de Total da Venda (Itens vs Cabec...,Aprovado,0


### 14 - Descrições dos dados 

In [13]:
# --- Etapa 4: Descrição dos Dados (Estatística Descritiva) ---
# Lista de tabelas que desejamos analisar
tabelas_para_analise = ['orders', 'order_items', 'customers'] 

print("Iniciando análise descritiva das tabelas...")

for tabela in tabelas_para_analise:
    try:
        # Chamada da nova função do seu arquivo de funções auxiliares
        omni_val.gerar_descricao_estatistica(con, tabela)
    except Exception as e:
        print(f"Erro ao analisar tabela {tabela}: {e}")

Iniciando análise descritiva das tabelas...

=== Resumo Estatístico: Tabela ORDERS ===

[Dados Numéricos]


,order_id,customer_id,free_shipping,shipping_cost,gross_amount,discount,net_amount
count,48193.000000,48193.000000,48193.0,48193.000000,48193.000000,48193.000000,48193.000000
mean,24097.000000,29349.595564,0.0,6.460125,181.239084,19.106666,168.592543
std,13912.265098,12682.008178,0.0,2.044032,141.178468,32.216407,126.752804
min,1.000000,1.000000,0.0,5.000000,22.200000,0.000000,20.540000
25%,12049.000000,19826.000000,0.0,5.000000,79.620000,0.000000,78.220000
50%,24097.000000,29389.000000,0.0,5.000000,141.900000,0.000000,130.450000
75%,36145.000000,39865.000000,0.0,9.100000,238.500000,27.790000,217.500000
max,48193.000000,51241.000000,0.0,11.200000,1453.500000,293.760000,1252.100000



[Dados Categóricos / Texto]


,order_date,shipped_date,delivery_date,order_status,payment_method,transaction_id,action_code
count,48193,48193,48193,48193,48193,48193,48193
unique,728,728,738,3,3,48193,37
top,2025-11-24,2025-11-25,2025-11-24,completed,credit_card,TXN-001-8EDD1F5C,BAU-GOOGLE-SALES
freq,269,269,266,46303,29060,1,9109


--------------------------------------------------

=== Resumo Estatístico: Tabela ORDER_ITEMS ===

[Dados Numéricos]


...

--------------------------------------------------

=== Resumo Estatístico: Tabela CUSTOMERS ===

[Dados Numéricos]


,customer_id
count,51241.000000
mean,25621.000000
std,14792.146909
min,1.000000
25%,12811.000000
50%,25621.000000
75%,38431.000000
max,51241.000000



[Dados Categóricos / Texto]


,first_name,last_name,email,gender,birth_date,city,state,signup_date
count,51241,51241,51241,51241,51241,51241,51241,51241
unique,283,102,14224,2,15408,526,27,1296
top,Ana,Lima,bruno.ferreira@email.com,F,1976-11-22,São Paulo,SP,2025-11-24
freq,2710,3269,260,39953,13,6270,15040,269


--------------------------------------------------
